# Demand forecast — seasonal outlook with holdout check

Monthly passenger demand → Prophet → 12-month forecast with confidence bands.

Path: **load → decompose → train → forecast → score holdout**.


In [ ]:
# %pip install prophet pandas matplotlib statsmodels

import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from prophet import Prophet

df = pd.read_csv("data/passengers.csv", parse_dates=["Month"])
df = df.sort_values("Month").reset_index(drop=True)

# Hold out the last 12 months (one seasonal cycle)
cutoff = df["Month"].max() - pd.DateOffset(months=11)
# Use explicit calendar split like the original design: 2023 as test if present
if (df["Month"] >= "2023-01-01").any():
    train = df[df["Month"] < "2023-01-01"].copy()
    test = df[df["Month"] >= "2023-01-01"].copy()
else:
    train = df[df["Month"] < cutoff].copy()
    test = df[df["Month"] >= cutoff].copy()

print(f"Train {train['Month'].min().date()} → {train['Month'].max().date()} (n={len(train)})")
print(f"Test  {test['Month'].min().date()} → {test['Month'].max().date()} (n={len(test)})")


## 1. What is the pattern?

In [ ]:
decompose = seasonal_decompose(
    train.set_index("Month")["Passengers"],
    model="additive",
    extrapolate_trend="freq",
    period=12,
)
decompose.plot()
plt.tight_layout()
plt.show()


## 2. Fit Prophet on train

In [ ]:
train_prophet = train.rename(columns={"Month": "ds", "Passengers": "y"})
model = Prophet()
model.fit(train_prophet)

future = model.make_future_dataframe(periods=len(test), freq="MS")
forecast = model.predict(future)

forecast_cols = ["ds", "yhat", "yhat_lower", "yhat_upper"]
print(forecast[forecast_cols].tail())


## 3. Score the holdout (what we actually validate)

In [ ]:
import numpy as np

merged = test.merge(forecast[forecast_cols], left_on="Month", right_on="ds", how="left")
merged["abs_err"] = (merged["Passengers"] - merged["yhat"]).abs()
merged["pct_err"] = merged["abs_err"] / merged["Passengers"]

mae = merged["abs_err"].mean()
rmse = float(np.sqrt(((merged["Passengers"] - merged["yhat"]) ** 2).mean()))
mape = merged["pct_err"].mean() * 100
coverage = ((merged["Passengers"] >= merged["yhat_lower"]) & (merged["Passengers"] <= merged["yhat_upper"])).mean()

print(f"MAE  = {mae:.2f}")
print(f"RMSE = {rmse:.2f}")
print(f"MAPE = {mape:.1f}%")
print(f"Interval coverage = {coverage:.0%} (of holdout months)")

merged[["Month", "Passengers", "yhat", "yhat_lower", "yhat_upper"]].round(1)


## 4. Chart — train, forecast band, real test months

In [ ]:
fig = model.plot(forecast)
ax = fig.gca()
train_end = train["Month"].max()
ax.axvline(x=train_end, color="red", linestyle="--", label="Train end")
ax.plot(test["Month"], test["Passengers"], "ro", markersize=4, label="Actual (test)")
ax.legend()
plt.show()
